# AEZ-26 Phase 2A — Daily Hydroclimate Backbone

This notebook is the reproducible Phase 2A workflow.

It:
- loads the three authoritative daily Phase-1 files,
- checks their calendars,
- merges them by exact one-to-one date,
- runs structural/scientific QC,
- automatically creates `Phase2/Phase2A/`,
- saves the backbone CSV and QC CSV there.

It performs **no interpolation, imputation, resampling, normalization, scaling, drought-index calculation, or event labeling**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path('/content/drive/MyDrive/AEZ26_Final_Research_Data')
PHASE2_DIR = BASE_DIR / 'Phase2'
PHASE2A_DIR = PHASE2_DIR / 'Phase2A'

PHASE2A_DIR.mkdir(parents=True, exist_ok=True)

CHIRPS_FILE = BASE_DIR / 'AEZ26_CHIRPS_Daily_Precipitation_2000_2025.csv'
ERA5_FILE   = BASE_DIR / 'AEZ26_ERA5Land_Daily_Comprehensive_2000_2025.csv'
GLDAS_FILE  = BASE_DIR / 'AEZ26_GLDAS_Daily_SM_Layers_Runoff_Baseflow_ET_2000_2025.csv'

print('Phase 2A output folder:', PHASE2A_DIR)

for p in [CHIRPS_FILE, ERA5_FILE, GLDAS_FILE]:
    print(p.name, '->', p.exists())
    if not p.exists():
        raise FileNotFoundError(f'Authoritative input not found: {p}')


In [ ]:
chirps = pd.read_csv(CHIRPS_FILE, parse_dates=['date'])
era5   = pd.read_csv(ERA5_FILE, parse_dates=['date'])
gldas  = pd.read_csv(GLDAS_FILE, parse_dates=['date'])

# Preserve QC provenance before merging.
chirps = chirps.rename(columns={
    'valid_pixel_count': 'CHIRPS_valid_pixel_count',
    'valid_pixel_fraction': 'CHIRPS_valid_pixel_fraction',
})

era5 = era5.rename(columns={
    'complete_day_flag': 'ERA5_complete_day_flag',
})

gldas = gldas.rename(columns={
    'state_complete_flag': 'GLDAS_state_complete_flag',
    'flux_complete_flag': 'GLDAS_flux_complete_flag',
    'complete_day_flag': 'GLDAS_complete_day_flag',
})

for df in (chirps, era5, gldas):
    df.sort_values('date', inplace=True)
    df.reset_index(drop=True, inplace=True)

print('CHIRPS:', chirps.shape)
print('ERA5-Land:', era5.shape)
print('GLDAS:', gldas.shape)


In [ ]:
expected_dates = pd.date_range('2000-01-01', '2025-12-31', freq='D')
assert len(expected_dates) == 9497

for name, df in [('CHIRPS', chirps), ('ERA5-Land', era5), ('GLDAS', gldas)]:
    assert len(df) == 9497, f'{name}: unexpected row count'
    assert df['date'].notna().all(), f'{name}: missing date'
    assert df['date'].is_unique, f'{name}: duplicate dates'
    assert pd.DatetimeIndex(df['date']).equals(expected_dates), f'{name}: calendar mismatch'

assert chirps['date'].equals(era5['date'])
assert chirps['date'].equals(gldas['date'])

print('All three source calendars are identical: PASS')


In [ ]:
daily = (
    chirps
    .merge(era5, on='date', how='inner', validate='one_to_one')
    .merge(gldas, on='date', how='inner', validate='one_to_one')
)

assert len(daily) == 9497
assert daily['date'].is_unique
assert pd.DatetimeIndex(daily['date']).equals(expected_dates)

print('Merged shape:', daily.shape)
print('Date range:', daily['date'].min().date(), 'to', daily['date'].max().date())


In [ ]:
qc = {}

qc['rows'] = len(daily)
qc['columns'] = daily.shape[1]
qc['duplicate_dates'] = int(daily['date'].duplicated().sum())
qc['missing_calendar_dates'] = len(
    expected_dates.difference(pd.DatetimeIndex(daily['date']))
)
qc['total_missing_cells'] = int(daily.isna().sum().sum())

qc['negative_precip_days'] = int(
    (daily['precipitation_mm_day'] < 0).sum()
)

qc['ERA5_hourly_count_not_24'] = int(
    (daily['number_of_hourly_ERA5_images_used'] != 24).sum()
)
qc['ERA5_daily_count_not_1'] = int(
    (daily['number_of_daily_ERA5_images_used'] != 1).sum()
)
qc['ERA5_incomplete_days'] = int(
    (daily['ERA5_complete_day_flag'] != 1).sum()
)

temp_ok = (
    (daily['t2m_max_C'] >= daily['t2m_mean_C']) &
    (daily['t2m_mean_C'] >= daily['t2m_min_C'])
)
qc['ERA5_temperature_order_violations'] = int((~temp_ok).sum())

qc['ERA5_RH_outside_0_100'] = int(
    (
        (daily['relative_humidity_mean_pct'] < 0) |
        (daily['relative_humidity_mean_pct'] > 100)
    ).sum()
)
qc['ERA5_negative_VPD_mean'] = int(
    (daily['VPD_mean_kPa'] < 0).sum()
)
qc['ERA5_VPDmax_below_mean'] = int(
    (daily['VPD_max_kPa'] < daily['VPD_mean_kPa']).sum()
)

qc['GLDAS_state_count_not_8'] = int(
    (daily['number_of_subdaily_state_images_used'] != 8).sum()
)
qc['GLDAS_flux_count_not_8'] = int(
    (daily['number_of_subdaily_flux_images_used'] != 8).sum()
)
qc['GLDAS_state_incomplete_days'] = int(
    (daily['GLDAS_state_complete_flag'] != 1).sum()
)
qc['GLDAS_flux_incomplete_days'] = int(
    (daily['GLDAS_flux_complete_flag'] != 1).sum()
)

sm040_err = (
    daily['SM_0_10cm_mm']
    + daily['SM_10_40cm_mm']
    - daily['SM_0_40cm_mm']
).abs()

sm0100_err = (
    daily['SM_0_40cm_mm']
    + daily['SM_40_100cm_mm']
    - daily['SM_0_100cm_mm']
).abs()

sm0200_err = (
    daily['SM_0_100cm_mm']
    + daily['SM_100_200cm_mm']
    - daily['SM_0_200cm_mm']
).abs()

runoff_err = (
    daily['surface_runoff_mm_day']
    + daily['baseflow_subsurface_mm_day']
    - daily['total_runoff_mm_day']
).abs()

qc['max_SM_0_40_identity_error_mm'] = float(sm040_err.max())
qc['max_SM_0_100_identity_error_mm'] = float(sm0100_err.max())
qc['max_SM_0_200_identity_error_mm'] = float(sm0200_err.max())
qc['max_runoff_identity_error_mm_day'] = float(runoff_err.max())

qc_df = pd.DataFrame(
    [{'check': k, 'value': v} for k, v in qc.items()]
)

display(qc_df)


In [ ]:
print('Known GLDAS edge case:')
display(
    daily.loc[
        daily['number_of_subdaily_state_images_used'] != 8,
        [
            'date',
            'number_of_subdaily_state_images_used',
            'GLDAS_state_complete_flag',
            'GLDAS_complete_day_flag'
        ]
    ]
)


## Save Phase 2A outputs automatically

The raw backbone is intentionally saved before any drought indices, anomalies, percentiles or ML preprocessing are created.


In [ ]:
BACKBONE_OUT = PHASE2A_DIR / 'AEZ26_Daily_Hydroclimate_Backbone_2000_2025.csv'
QC_OUT = PHASE2A_DIR / 'AEZ26_Phase2A_Daily_Backbone_QC.csv'

daily.to_csv(BACKBONE_OUT, index=False, date_format='%Y-%m-%d')
qc_df.to_csv(QC_OUT, index=False)

print('SAVED AUTOMATICALLY:')
print('Backbone ->', BACKBONE_OUT)
print('QC       ->', QC_OUT)

print()
print('FINAL CHECK')
print('Rows:', len(daily))
print('Columns:', daily.shape[1])
print('Expected merged shape: (9497, 38)')


## Phase 2A complete

Do not overwrite this raw backbone in later phases.

The next methodological stage is the literature-locked flash-drought definition, followed by pentad/event detection.
